In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score, accuracy_score
from sklearn.preprocessing import LabelBinarizer

def preprocess(df):

    df.dropna(thresh=df.shape[0]*0.5, axis=1, inplace=True)
    df.fillna(df.mean(), inplace=True)

    non_numeric_cols = df.select_dtypes(exclude=['int', 'float']).columns
    for col in non_numeric_cols:
        df[col].fillna(df[col].mode()[0], inplace=True)

    df.columns = [str(i) for i in range(df.shape[1])]
    ordinal_encoder = OrdinalEncoder()
    df.iloc[:, :-1] = ordinal_encoder.fit_transform(df.iloc[:, :-1])

    return df

def evaluate(X_train, X_test, y_train, y_test):
    #model1 = XGBClassifier(use_label_encoder=False, eval_metric='error', subsample=1.0, n_estimators=100, max_depth=5, learning_rate=0.1, colsample_bytree=0.5)
    #model2 = RandomForestClassifier(n_estimators=400, min_samples_split=5, min_samples_leaf=4, max_depth=None, bootstrap=True, class_weight={0: 1, 1: 100})
    #model3 = ExtraTreesClassifier(n_estimators=200, min_samples_split=5, min_samples_leaf=4, max_depth=30, bootstrap=False)
    model4 = GradientBoostingClassifier(subsample=0.7, n_estimators=400, max_depth=3, learning_rate=0.1)

    model_name_list = ['XGB Classifier', 'Random Forest', 'Extra Trees', 'Gradient Boosted']
    results = pd.DataFrame(columns=['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'cohen_kappa'], index=model_name_list)

    for i, model in enumerate([model4]):
        print(model)
        model.fit(X_train, y_train)
        test_probs = model.predict_proba(X_test)[:, 1]

        # Apply threshold of 0.3 to get the binary predictions
        test_predictions = np.where(test_probs > 0.3, 1, 0)
        print(test_predictions)


        #df_res = pd.DataFrame(test_predictions, columns=['kaggle_id','target'],index=False)
        if model == model4:

            name = '/users/beril/desktop/data/' +model_name_list[i] + '.csv'
            df_res = pd.DataFrame(test_predictions, columns = ['target'])
            df_res.index += 1  
            df_res.index.names = ['kaggle_id']
            df_res.to_csv(name, index=True)

        lb = LabelBinarizer()
        y_test_bin = lb.fit_transform(y_test)

        accuracy = accuracy_score(y_test, test_predictions)
        precision = precision_score(y_test, test_predictions, average='weighted', zero_division=0)
        recall = recall_score(y_test, test_predictions, average='weighted', zero_division=0)
        f1 = f1_score(y_test, test_predictions, average=None)
        _auc = roc_auc_score(y_test_bin, test_predictions)
        kappa = cohen_kappa_score(y_test, test_predictions)

        results.loc[model_name_list[i]] = [accuracy, precision, recall, f1, _auc, kappa]

    return results

df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

df_train = preprocess(df_train)
df_test = preprocess(df_test)

X_train = df_train.iloc[:, :-1].values  
X_train = OrdinalEncoder().fit_transform(X_train).astype('int')
y_train = df_train.iloc[:, -1]

X_test = df_test.iloc[:, :-1].values  
X_test = OrdinalEncoder().fit_transform(X_test).astype('int')
y_test = df_test.iloc[:, -1]

# Reshape y_train and y_test to 2D arrays
#y_train = y_train.values.reshape(-1, 1)
#y_test = y_test.values.reshape(-1, 1)

evaluation_results = evaluate(X_train, X_test, y_train, y_test)
print(evaluation_results)